# Rebuild corrected POPS and UHSAS lookup tables

Run this notebook from top to bottom in the **Research** environment. It builds POPS first, then UHSAS, with eight worker processes. It does not start aerosol merge production or replace the historical packaged tables.

- Corrected solid-angle and polarization calculation: `solid-angle-polarized-cones-v1`.
- POPS: mirror-only, 52° collection half-angle, 405 nm.
- UHSAS: one collection arm, 14.8–57° annular opening, 1054 nm.
- Preserve the historical diameter and real-index grids. Retain all 31 historical absorption-index values and add **0.001 exactly**, so the POPS source index is a grid point.
- Write new tables, per-instrument logs, settings, and verification reports into the explicit directory below.
- Existing tables or an existing run-settings file cause a stop. To rebuild again, choose a new output directory.

The numerical checks below do not establish agreement with calibration measurements or validate a new campaign product. See [optical model notes](../docs/optical_model.md).


In [ ]:
from pathlib import Path
from dataclasses import asdict
from datetime import datetime, timezone
from contextlib import redirect_stdout
from importlib.metadata import version
import hashlib
import json
import os
import subprocess
import sys
import time

# Avoid nested numerical-library thread pools inside the worker processes.
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"
os.environ["MIEPYTHON_USE_JIT"] = "1"

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src/sizedistmerge").is_dir():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "src/sizedistmerge").is_dir(), "Run inside the SizeDistMerge repository."
sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import zarr
from joblib import parallel_config
from sizedistmerge import optical_diameter as od


## Settings
All numerical settings and output destinations are visible here. The 1,000 diameter grid points are not the 100 response groups later used during diameter conversion.


In [ ]:
BUILD_ROOT = Path("/Users/C832577250/Output/sizedistmerge_optical_luts_solid_angle_20260911")
EXPECTED_MODEL = "solid-angle-polarized-cones-v1"
WORKERS = 8
CHUNKS = (128, 64, 1)
N_RANGE = (1.30, 1.80, 0.0005)

# k is the imaginary (absorbing) part of the particle refractive index.
LEGACY_K = np.array([
    0., 0.0001, 0.000136, 0.000186, 0.000253, 0.000345, 0.000471,
    0.000642, 0.000875, 0.001193, 0.001627, 0.002218, 0.003023,
    0.004122, 0.005619, 0.00766, 0.010443, 0.014237, 0.01941,
    0.026461, 0.036074, 0.04918, 0.067046, 0.091404, 0.12461,
    0.16988, 0.231597, 0.315735, 0.430439, 0.586814, 0.8,
])
K_VALUES = np.unique(np.r_[LEGACY_K, 0.001])
SPECS = {
    "pops": {
        "filename": "pops_sigma_col_405nm.zarr",
        "D_range": (60., 6000., 1000),
        "wavelength_nm": 405.,
        "geometry": od.POPSGeom(ring_step_deg=0.25, pmt_aperture_d_mm=0.),
        "source_ri": complex(1.615, 0.001),
    },
    "uhsas": {
        "filename": "uhsas_sigma_col_1054nm.zarr",
        "D_range": (30., 6000., 1000),
        "wavelength_nm": 1054.,
        "geometry": od.UHSASGeom(ring_step_deg=0.25),
        "source_ri": complex(1.52, 0.),
    },
}
LUT_PATHS = {name: BUILD_ROOT / spec["filename"] for name, spec in SPECS.items()}
OPTICAL_FILE = REPO_ROOT / "src/sizedistmerge/optical_diameter.py"

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def optical_sha256():
    return hashlib.sha256(OPTICAL_FILE.read_bytes()).hexdigest()


## Check settings and protect existing data
Read only the old coordinate arrays and metadata—not their calculated cross-sections. The output tables must be new. Settings record the exact optical-code hash and package versions.


In [ ]:
assert od.OPTICAL_MODEL_VERSION == EXPECTED_MODEL
assert od.RI_POPS_SRC == SPECS["pops"]["source_ri"]
assert od.RI_UHSAS_SRC == SPECS["uhsas"]["source_ri"]
assert WORKERS <= (os.cpu_count() or 1)
assert len(K_VALUES) == 32 and 0.001 in K_VALUES

for name, spec in SPECS.items():
    old = zarr.open_group(REPO_ROOT / "lut" / spec["filename"], mode="r")
    d_min, d_max, d_count = spec["D_range"]
    np.testing.assert_allclose(old["coords/D_nm"][:], np.geomspace(d_min, d_max, d_count), rtol=1e-12)
    np.testing.assert_allclose(old["coords/n"][:], np.arange(N_RANGE[0], N_RANGE[1]+1e-12, N_RANGE[2]), rtol=1e-12)
    np.testing.assert_array_equal(old["coords/k"][:], LEGACY_K)
    assert old.attrs["wavelength_nm"] == spec["wavelength_nm"]
    assert not LUT_PATHS[name].exists(), f"Output already exists: {LUT_PATHS[name]}"
assert od.pops_geometry_cache(SPECS["pops"]["geometry"]).direct is None
od.uhsas_geometry_cache(SPECS["uhsas"]["geometry"])

SOURCE_HASH = optical_sha256()
settings = {
    "created_utc": utc_now(),
    "model_version": EXPECTED_MODEL,
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True).strip(),
    "optical_code_sha256": SOURCE_HASH,
    "python": sys.executable,
    "packages": {name: version(name) for name in ("numpy", "miepython", "zarr", "joblib", "scipy", "scikit-learn")},
    "workers": WORKERS, "parallel_backend": "loky processes",
    "n_range": N_RANGE, "k_values": K_VALUES.tolist(), "chunks": CHUNKS,
    "instruments": {
        name: {"path": str(LUT_PATHS[name]), "D_range": spec["D_range"],
               "wavelength_nm": spec["wavelength_nm"], "geometry": asdict(spec["geometry"]),
               "source_ri": [spec["source_ri"].real, spec["source_ri"].imag]}
        for name, spec in SPECS.items()
    },
}
BUILD_ROOT.mkdir(parents=True, exist_ok=True)
with (BUILD_ROOT / "run_settings.json").open("x") as handle:
    json.dump(settings, handle, indent=2)

state = {"started_utc": utc_now(), "pid": os.getpid(), "model_version": EXPECTED_MODEL,
         "instruments": {name: {"state": "queued", "path": str(LUT_PATHS[name])} for name in SPECS}}
def save_status():
    state["updated_utc"] = utc_now()
    temp = BUILD_ROOT / "build_status.tmp"
    temp.write_text(json.dumps(state, indent=2))
    temp.replace(BUILD_ROOT / "build_status.json")

save_status()
print(json.dumps(settings, indent=2))


## Build and verify one instrument

The existing library builder does the optical calculation; this notebook supplies explicit settings and saves progress. Worker processes calculate different refractive indices. Each writes through the parent builder, avoiding simultaneous writes to the same table.

A table is marked **verified** here only after its grids, completion marker, finite positive values, selected direct calculations, and unchanged-index diameter conversions pass. The optical-code hash must also remain unchanged during the build.


In [ ]:
class BuildLog:
    """Send the builder's printed progress to both the notebook and a log."""
    def __init__(self, console, log):
        self.console, self.log = console, log
    def write(self, text):
        self.console.write(text)
        self.log.write(text)
        self.flush()
        return len(text)
    def flush(self):
        self.console.flush()
        self.log.flush()

def verify_table(name):
    spec = SPECS[name]
    root = zarr.open_group(LUT_PATHS[name], mode="r")
    assert root.attrs["optical_model_version"] == EXPECTED_MODEL
    assert root.attrs["build_complete"] is True
    assert root.attrs["wavelength_nm"] == spec["wavelength_nm"]
    assert root.attrs["collection_arms"] == 1
    if name == "pops":
        assert root.attrs["direct_collection"] is False
    for key, value in asdict(spec["geometry"]).items():
        assert root.attrs[key] == value, (key, root.attrs[key], value)

    d = root["coords/D_nm"][:]
    n = root["coords/n"][:]
    k = root["coords/k"][:]
    np.testing.assert_allclose(d, np.geomspace(*spec["D_range"]), rtol=1e-12)
    np.testing.assert_allclose(n, np.arange(N_RANGE[0], N_RANGE[1]+1e-12, N_RANGE[2]), rtol=1e-12)
    np.testing.assert_array_equal(k, K_VALUES)
    sig = root["sigma_col"]
    assert sig.shape == (len(d), len(n), len(k))
    for ik in range(len(k)):
        values = sig[:, :, ik]
        assert np.all(np.isfinite(values) & (values > 0)), f"Invalid cross-sections at k={k[ik]}"

    # Independently recompute selected table entries with the same reviewed kernel.
    fn = od.pops_csca if name == "pops" else od.uhsas_csca
    di = [0, 249, 499, 749, 999]
    largest_relative_error = 0.
    for target_n in (1.3, spec["source_ri"].real, 1.8):
        ni = int(np.argmin(abs(n - target_n)))
        for target_k in (0., 0.001, 0.8):
            ki = int(np.argmin(abs(k - target_k)))
            expected = fn(d[di], complex(n[ni], k[ki]), spec["wavelength_nm"], geom=spec["geometry"])
            saved = np.asarray([sig[j, ni, ki] for j in di])
            np.testing.assert_allclose(saved, expected, rtol=1e-6, atol=0.)
            largest_relative_error = max(largest_relative_error, float(np.max(abs(saved/expected - 1))))

    lut = od.SigmaLUT(str(LUT_PATHS[name]))
    query_d = np.geomspace(100., 3000. if name == "pops" else 1000., 100)
    converted = od.convert_do_lut(query_d, spec["source_ri"], spec["source_ri"], lut, response_bins=100)
    np.testing.assert_allclose(converted, query_d, rtol=1e-10, atol=0.)
    assert optical_sha256() == SOURCE_HASH, "Optical code changed during this build."
    report = {"verified_utc": utc_now(), "shape": list(sig.shape),
              "all_finite_positive": True, "direct_check_points": 45,
              "largest_direct_relative_error": largest_relative_error,
              "largest_identity_relative_error": float(np.max(abs(converted/query_d-1))),
              "optical_code_sha256": SOURCE_HASH,
              "scope": "Numerical table checks only; calibration and campaign comparisons remain pending."}
    (BUILD_ROOT / f"{name}_verification.json").write_text(json.dumps(report, indent=2))
    return report

def build_instrument(name):
    spec = SPECS[name]
    assert optical_sha256() == SOURCE_HASH, "Optical code changed since preflight."
    assert not LUT_PATHS[name].exists(), f"Refusing to overwrite {LUT_PATHS[name]}"
    start = time.perf_counter()
    state["instruments"][name].update(state="building", started_utc=utc_now())
    save_status()
    try:
        with (BUILD_ROOT / f"{name}_build.log").open("x", buffering=1) as log:
            with redirect_stdout(BuildLog(sys.stdout, log)):
                print(f"{name.upper()} started at {utc_now()}", flush=True)
                with parallel_config(backend="loky", inner_max_num_threads=1):
                    od.build_sigma_lut(
                        str(LUT_PATHS[name]), name, spec["wavelength_nm"], spec["geometry"],
                        D_range=spec["D_range"], n_range=N_RANGE, k_values=K_VALUES,
                        chunks=CHUNKS, jobs_per_k=WORKERS, parallel_backend="processes",
                    )
        state["instruments"][name].update(state="verifying", calculation_finished_utc=utc_now())
        save_status()
        report = verify_table(name)
        state["instruments"][name].update(state="verified", finished_utc=utc_now(),
                                          elapsed_seconds=time.perf_counter()-start)
        save_status()
        print(name.upper(), "verified:", json.dumps(report, indent=2))
    except BaseException as error:
        state["instruments"][name].update(state="failed", failed_utc=utc_now(), error=repr(error))
        save_status()
        raise


## Build POPS
Mirror-only; no direct detector path is included. The source calibration index is 1.615 + 0.001i.


In [ ]:
build_instrument("pops")


## Build UHSAS
This starts automatically after POPS finishes and passes its checks. The source calibration index is 1.52 + 0i.


In [ ]:
build_instrument("uhsas")


## Final status
The table paths below can be used for the next comparison. This notebook does not change the production notebook's LUT directory or launch a merge.


In [ ]:
assert all(item["state"] == "verified" for item in state["instruments"].values())
state["finished_utc"] = utc_now()
save_status()
for name, path in LUT_PATHS.items():
    print(f"{name.upper()}: {path}")
print("Both corrected LUTs are built and numerically checked. Merge production remains stopped.")
